# Quickstart

This page walks through three complete examples: solving the full drift kinetic
equation (DKE) for one or more species, finding the ambipolar radial electric field,
and solving the simplified monoenergetic drift kinetic equation (MDKE) to get a 3×3
transport matrix.

This page is a Jupyter notebook, so you can run and edit each example yourself. The
resolutions used are deliberately low so that all of it runs in a few minutes on a
CPU; the results are qualitatively right but not converged. See
[Performance](performance.rst) for how to choose resolutions for real work.

In [1]:
# Setup for running this page: the example equilibria live in tests/data
import os
from pathlib import Path

for _d in [Path.cwd(), *Path.cwd().parents]:
    if (_d / "tests" / "data").is_dir():
        os.chdir(_d / "tests" / "data")
        break

## Radial coordinate convention

yancc uses a single radial coordinate everywhere:

$$
\rho = \sqrt{s} = \sqrt{\psi_t / \psi_{t,\mathrm{LCFS}}},
$$

i.e. the square root of the normalized toroidal flux. This is the only radial
variable that appears in the public API.

This differs from several other codes:

- Some codes (e.g. VMEC, BOOZ_XFORM) parameterize surfaces by
  $s = \rho^2$, the normalized toroidal flux itself.
- Others use the dimensional minor radius $r = \rho\, a$, where
  $a$ is the minor radius of the LCFS.
- Many codes mix conventions, e.g. labelling surfaces by $s$ but
  expressing gradients in $r$.

In yancc, **every** radial input and output is in $\rho$. In particular:

- [Field](_api/yancc.Field.rst) constructors (`from_desc`, `from_vmec`, `from_booz_xform`, `from_ipp_bc`,
  `from_boozer`) all take `rho`, not `s` or `r`. If your input is $s$, pass
  `rho = sqrt(s)`.
- The radial electric field passed to [solve_dke](_api/yancc.solve_dke.rst) is
  $E_\rho = -\partial \Phi / \partial \rho$, in Volts. If you have
  $E_r = -\partial \Phi / \partial r$ (in V/m), multiply by
  `field.a_minor` to convert.
- The density and temperature gradients on [LocalMaxwellian](_api/yancc.LocalMaxwellian.rst) are
  $\partial n / \partial \rho$ and $\partial T / \partial \rho$.
  If you have $dn/dr$ and $dT/dr$, multiply by `field.a_minor`.
- The monoenergetic drive `erhohat` passed to [solve_mdke](_api/yancc.solve_mdke.rst) is
  $E_\rho / v$ in V·s/m, again using $\rho$ (not $r$ or $s$).

Make sure all inputs are converted to the $\rho$ convention before
calling into yancc; otherwise gradients will silently disagree with what the
solver expects, by factors of $a$ or $2\rho$.

## Solving the Full DKE

The full DKE requires a magnetic field, pitch-angle and speed grids, one or
more species, and a radial electric field. Each species is a
[LocalMaxwellian](_api/yancc.LocalMaxwellian.rst) built from a
[Species](_api/yancc.Species.rst) (for example one of the predefined isotope
constants like `Hydrogen`).

In [2]:
from yancc import (
    Field,
    Hydrogen,
    LocalMaxwellian,
    MaxwellSpeedGrid,
    UniformPitchAngleGrid,
    solve_dke,
)

# Field and grids. These resolutions are low so that this page runs quickly.
rho = 0.5
nt, nz, na, nx = 9, 15, 25, 4
field = Field.from_vmec("wout_NCSX.nc", rho, nt, nz)
pitchgrid = UniformPitchAngleGrid(na)
speedgrid = MaxwellSpeedGrid(nx)

In [3]:
# Single hydrogen species. Density and temperature gradients are with
# respect to rho = sqrt(normalized toroidal flux), so multiply physical
# gradients by the minor radius.
species = [
    LocalMaxwellian(
        Hydrogen,
        temperature=0.8e3,                        # eV
        density=1.5e20,                           # 1/m^3
        dTdrho=-2.0e3 * field.a_minor,
        dndrho=-0.4e20 * field.a_minor,
    )
]

# Radial electric field, in Volts. Erho = -dPhi/drho.
Er_kV_per_m = 0.5
Erho = Er_kV_per_m * field.a_minor * 1000

In [4]:
sol, info = solve_dke(
    field,
    pitchgrid,
    speedgrid,
    species,
    Erho=Erho,
    verbose=1,
    rtol=1e-5,
)

Field info (source: vmec):
    ρ         =  0.500              ι         = -4.689e-01
    <B>       =  1.538e+00 T        δ_B       =  4.857e-02
    Bmax/Bmin =  1.177e+00          f_trapped =  3.858e-01
    I         = -3.374e-03 T·m      G         =  2.320e+00 T·m
Species  0:  m= 1.00e+00 (mₚ)      q= 1.00e+00 (qₚ)
             n= 1.50e+20 (m⁻³)  a/Lₙ= 8.60e-02
             T= 8.00e+02 (eV)   a/Lᴛ= 8.07e-01  
             ν* (x=1.34e-01):  2.413e+01
             ν* (x=1.00e+00):  4.847e-02
             ν* (x=2.26e+00):  2.654e-03
<E||B> :  0.00e+00 (V*T/m)
Eᵨ = -∂Φ /∂ρ:  1.61e+02 (V)
E* (x=1.0): [ 8.305e-04 ] (per species)
Mₚ (x=1.0): [ 8.288e-03 ] (per species)
A₁: [ 9.222e-01 ] (per species)
A₂: [-8.066e-01 ] (per species)
A₃: [ 0.000e+00 ] (per species)
Grid 0: nx=   4, nα=  21, nθ=   8, nζ=  13, N=8,736
Grid 1: nx=   4, nα=  25, nθ=   9, nζ=  15, N=13,500
Finished krylov: nmv=  17, n_restarts=  1, residual=6.741e-06
<heat_flux>    : [ 5.375e+05 ] (per species, kg·m⁻¹·s⁻³ = W·m⁻³)

In [5]:
# Common moments. See the list of variables for all the available quantities.
print("<Gamma>  =", sol.get("<particle_flux>"))   # particles/(m^2 s)
print("<Q>      =", sol.get("<heat_flux>"))       # J/(m^2 s)
print("<V||B>   =", sol.get("<V||B>"))            # T*m/s
print("<J||B>   =", sol.get("<J||B>"))            # T*A/m^2

<Gamma>  = [6.87278277e+20]
<Q>      = [537456.50236449]
<V||B>   = [-44960.69138001]
<J||B>   = -1080524.5376630418


The `sol` object is a [DKESolution](_api/yancc.DKESolution.rst); pass any name
listed in [variables](variables.rst) to `sol.get(...)`.

### Multiple species and background species

`species` can contain several entries to solve coupled species
simultaneously. The optional `background` argument adds species to the
collision operator without solving for their perturbed distribution.

Rather than building each [LocalMaxwellian](_api/yancc.LocalMaxwellian.rst) by hand,
it is often more convenient to define radial profiles once as a
[GlobalMaxwellian](_api/yancc.GlobalMaxwellian.rst) and let
[GlobalMaxwellian.localize](_api/yancc.GlobalMaxwellian.rst#yancc.GlobalMaxwellian.localize) evaluate the temperature,
density, and their $\rho$-gradients at the surface of interest:

In [6]:
import jax.numpy as jnp
from yancc import Electron, GlobalMaxwellian, Hydrogen

# Profiles in rho = sqrt(normalized toroidal flux).
def T_profile(rho):
    return 0.8e3 * (1.0 - rho**2)        # eV

def n_profile(rho):
    return 1.5e20 * (1.0 - rho**2)       # 1/m^3

globals_ = [
    GlobalMaxwellian(Hydrogen, T_profile, n_profile),
    GlobalMaxwellian(Electron, T_profile, n_profile),
]

# localize(rho) returns a LocalMaxwellian with dTdrho, dndrho filled in
# automatically via JAX autodiff of the profile callables.
species = [g.localize(rho) for g in globals_]

sol, info = solve_dke(
    field, pitchgrid, speedgrid, species,
    Erho=Erho,
    EparB=0.0,
    verbose=1,
)

Field info (source: vmec):
    ρ         =  0.500              ι         = -4.689e-01
    <B>       =  1.538e+00 T        δ_B       =  4.857e-02
    Bmax/Bmin =  1.177e+00          f_trapped =  3.858e-01
    I         = -3.374e-03 T·m      G         =  2.320e+00 T·m
Species  0:  m= 1.00e+00 (mₚ)      q= 1.00e+00 (qₚ)
             n= 1.12e+20 (m⁻³)  a/Lₙ= 1.33e+00
             T= 6.00e+02 (eV)   a/Lᴛ= 1.33e+00  
             ν* (x=1.34e-01):  3.230e+01
             ν* (x=1.00e+00):  6.514e-02
             ν* (x=2.26e+00):  3.618e-03
Species  1:  m= 5.45e-04 (mₚ)      q=-1.00e+00 (qₚ)
             n= 1.12e+20 (m⁻³)  a/Lₙ= 1.33e+00
             T= 6.00e+02 (eV)   a/Lᴛ= 1.33e+00  
             ν* (x=1.34e-01):  3.159e+02
             ν* (x=1.00e+00):  1.508e-01
             ν* (x=2.26e+00):  6.708e-03
<E||B> :  0.00e+00 (V*T/m)
Eᵨ = -∂Φ /∂ρ:  1.61e+02 (V)
E* (x=1.0): [ 9.589e-04  2.238e-05 ] (per species)
Mₚ (x=1.0): [ 9.570e-03  2.233e-04 ] (per species)
A₁: [ 3.978e-01  9.355e-01 ] (per 

### Finding the ambipolar electric field

In a stellarator the radial electric field is not a free parameter. It is set by
ambipolarity: the radial current $J_\rho = \sum_s q_s \Gamma_s$, summed over
species with charge $q_s$ and particle flux $\Gamma_s$, must vanish. For the
arbitrary `Erho` used above it generally does not:

In [7]:
charges = jnp.array([sp.species.charge for sp in species])   # C
Gamma = sol.get("<particle_flux>")                            # particles/(m^2 s)
print("<Gamma> per species  =", Gamma)
print("J_rho = sum_s q_s Gamma_s  =", (charges * Gamma).sum(), "A/m^3")

<Gamma> per species  = [7.97571973e+20 5.38711268e+19]
J_rho = sum_s q_s Gamma_s  = 119.15401183808945 A/m^3


[solve_dke_ambipolar](_api/yancc.solve_dke_ambipolar.rst) searches for the values of `Erho` where $J_\rho = 0$,
solving the DKE at each trial value. It takes the same field, grids and species as
[solve_dke](_api/yancc.solve_dke.rst), plus `num_roots`, the number of roots to look for. There can be
more than one root, for example an ion root and an electron root, and the search uses
deflation to avoid converging to the same one twice. By default it searches
$|E_\rho|$ up to a normalized field $E^* = E_\rho / (a\, v_{th} \langle B \rangle) = 0.1$ for the
first species; pass `bounds` to narrow that, or to look elsewhere.

In [8]:
from yancc import solve_dke_ambipolar

Erho_root, sols, info_root = solve_dke_ambipolar(
    field,
    pitchgrid,
    speedgrid,
    species,
    num_roots=1,
    verbose=1,
)

Field info (source: vmec):
    ρ         =  0.500              ι         = -4.689e-01
    <B>       =  1.538e+00 T        δ_B       =  4.857e-02
    Bmax/Bmin =  1.177e+00          f_trapped =  3.858e-01
    I         = -3.374e-03 T·m      G         =  2.320e+00 T·m
Species  0:  m= 1.00e+00 (mₚ)      q= 1.00e+00 (qₚ)
             n= 1.12e+20 (m⁻³)  a/Lₙ= 1.33e+00
             T= 6.00e+02 (eV)   a/Lᴛ= 1.33e+00  
             ν* (x=1.34e-01):  3.230e+01
             ν* (x=1.00e+00):  6.514e-02
             ν* (x=2.26e+00):  3.618e-03
Species  1:  m= 5.45e-04 (mₚ)      q=-1.00e+00 (qₚ)
             n= 1.12e+20 (m⁻³)  a/Lₙ= 1.33e+00
             T= 6.00e+02 (eV)   a/Lᴛ= 1.33e+00  
             ν* (x=1.34e-01):  3.159e+02
             ν* (x=1.00e+00):  1.508e-01
             ν* (x=2.26e+00):  6.708e-03
Grid 0: nx=   4, nα=  17, nθ=   6, nζ=  10, N=8,160
Grid 1: nx=   4, nα=  25, nθ=   9, nζ=  15, N=27,000
Search   0: start= 0.0000e+00, bounds=(-1.6823e+04, 1.6823e+04), success=True, x=-1.86

It returns the roots, a `DKESolution` at each one, and information about the search.
Roots that could not be found are set to `inf`, with `success=False`.

In [9]:
print("root found                         =", info_root["success"])
print("ambipolar E_rho [V]                =", Erho_root)
print("ambipolar E_r [kV/m]               =", Erho_root / field.a_minor / 1e3)
print("normalized radial current at root  =", info_root["residual"])
print("total matrix-vector products       =", info_root["nmv_total"])

# the solution at the root is a regular DKESolution
Gamma_root = sols[0].get("<particle_flux>")
print("<Gamma> per species at the root    =", Gamma_root)
print("J_rho at the root [A/m^3]          =", (charges * Gamma_root).sum())

root found                         = [ True]
ambipolar E_rho [V]                = [-1860.53519435]
ambipolar E_r [kV/m]               = [-5.76670461]
normalized radial current at root  = [6.34121868e-07]
total matrix-vector products       = 89
<Gamma> per species at the root    = [6.71002840e+19 6.70997741e+19]
J_rho at the root [A/m^3]          = 8.169203857910645e-05


See the [Advanced Tuning](tuning.rst) page for the options that control the search.

### Loading a field from other equilibria

[Field](_api/yancc.Field.rst) can be constructed from several equilibrium
formats. The same physical surface can be obtained from any of them. DESC is an
optional dependency, so that example is not run here:

```python
import desc

eq = desc.io.load("NCSX_output.h5")[-1]
field_desc = Field.from_desc(eq, rho, nt, nz)
```

In [10]:
rho = 0.5
nt, nz = 17, 37

field_vmec = Field.from_vmec("wout_NCSX.nc", rho, nt, nz)
field_booz = Field.from_booz_xform("boozmn_wout_NCSX.nc", rho, nt, nz)
field_bc   = Field.from_ipp_bc("NCSX.bc", rho, nt, nz)

See [Loading Fields](loading_fields.rst) or [Field](_api/yancc.Field.rst) for more information.

### JAX transformations

[solve_dke](_api/yancc.solve_dke.rst) and [solve_mdke](_api/yancc.solve_mdke.rst) are
implemented in JAX, so the standard transformations (`jax.jit`,
`jax.vmap`, `jax.jacfwd`, `jax.jacrev`, ...) can be applied to them in
the usual way.

## Solving the Monoenergetic DKE

The MDKE depends only on a magnetic field, a pitch-angle grid, and two
normalized scalars: a monoenergetic electric field `erhohat` and a
monoenergetic collisionality `nuhat`. The solution object exposes the monoenergetic
transport matrix via `sol.get("Dij")`.

In [11]:
from yancc import Field, UniformPitchAngleGrid, solve_mdke

# Magnetic field on a flux surface (here from a BOOZ_XFORM file).
nt, nz = 17, 33
rho = 0.200 ** 0.5      # rho = sqrt(s); pick the surface to load
field = Field.from_booz_xform(
    "boozmn_wout_w7x_eim.nc",
    rho,
    nt,
    nz,
    cutoff=1e-5,
)

# Uniform finite-difference grid in pitch angle.
pitchgrid = UniformPitchAngleGrid(65)

In [12]:
# Monoenergetic drives.
erhohat = 1.0e-4      # E_rho / v   [V*s/m]
nuhat = 1.0e-2        # nu / v      [1/m]

sol, info = solve_mdke(
    field,
    pitchgrid,
    erhohat,
    nuhat,
    verbose=1,
)

# 3x3 transport matrix.
Dij = sol.get("Dij")
print("D11 =", Dij[0, 0])
print("D31 =", Dij[2, 0])
print("D33 =", Dij[2, 2])

Field info (source: booz_xform):
    ρ         =  0.447              ι         = -8.621e-01
    <B>       =  2.422e+00 T        δ_B       =  4.367e-02
    Bmax/Bmin =  1.185e+00          f_trapped =  4.444e-01
    I         = -7.965e-18 T·m      G         =  1.409e+01 T·m
ν` =  1.000e-02
E` =  1.000e-04
Grid 0: nα=  39, nθ=  10, nζ=  20, N=7,800
Grid 1: nα=  65, nθ=  17, nζ=  33, N=36,465
Finished krylov (1st rhs): nmv=  11, n_restarts=  1, residual=6.651e-06
Finished krylov (2nd rhs): nmv=  10, n_restarts=  1, residual=5.672e-06
D11 = 0.012941627786515761
D31 = 0.1894734739066111
D33 = 342.59883455239657


The `sol` object is a [MDKESolution](_api/yancc.MDKESolution.rst); see
[variables](variables.rst) for the full list of names accepted by `sol.get(...)`.